# Tracing: MLflow OpenTelemetry-Compatible Tracing

This notebook demonstrates how the Custom Deep Research Lab leverages **MLflow autolog
tracing** to capture full execution traces of LangGraph agents. With a single line of
configuration, every LLM call, tool invocation, and agent decision is recorded as an
OpenTelemetry-compatible span tree \u2014 enabling deep debugging and performance analysis
without modifying application code.

## 1. Load Environment

In [ ]:
import os
from dotenv import load_dotenv

env_path = os.path.join(os.path.dirname(os.getcwd()), ".env")
load_dotenv(env_path)

mlflow_uri = os.getenv("MLFLOW_TRACKING_URI", "")
experiment = os.getenv("MLFLOW_EXPERIMENT_NAME", "deep-research-harness")
print(f"MLflow Tracking URI: {mlflow_uri or '(not set)'}")
print(f"Experiment: {experiment}")
if mlflow_uri:
    print("\u2705 Environment loaded")
else:
    print("\u26a0\ufe0f MLFLOW_TRACKING_URI not set \u2014 configure it in .env")

## 2. Connect to MLflow

In [ ]:
import mlflow

mlflow.set_tracking_uri(mlflow_uri)
mlflow.set_experiment(experiment)
exp = mlflow.get_experiment_by_name(experiment)
if exp:
    print(f"\u2705 Connected to experiment: {exp.name} (ID: {exp.experiment_id})")
else:
    print(f"\u2705 Created new experiment: {experiment}")

## 3. How Tracing Works

MLflow autolog tracing requires **zero code changes** to your LangGraph agents.
The entire setup is three lines in `backend/observability.py`:

```python
import mlflow
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI"))
mlflow.langchain.autolog()
```

Once enabled, MLflow automatically instruments:

| Component | What Gets Captured |
|-----------|-------------------|
| LangGraph execution | Full graph traversal path, node transitions |
| LLM calls | Model name, prompt, completion, token counts, latency |
| Tool invocations | Tool name, input arguments, output, duration |
| Agent decisions | Routing logic, conditional edges, state updates |

Each research session generates a single **trace** containing a tree of **spans** \u2014
one span per operation. This gives you full visibility into what the agent did and why.

## 4. Explore Traces in MLflow UI

Follow these steps to explore traces visually:

1. **Navigate to MLflow UI** \u2014 Open the RHOAI Dashboard and click the MLflow link
   (or use the route URL from `oc get route mlflow -n doc-research-lab`)
2. **Select the experiment** \u2014 Click on `deep-research-harness` in the left panel
3. **Click the Traces tab** \u2014 You will see a list of all recorded traces
4. **Click on a trace** \u2014 View the Summary (metadata), Details (span tree),
   and Timeline (Gantt chart of execution)
5. **Click individual spans** \u2014 Inspect tool inputs/outputs, LLM prompts and
   completions, token counts, and latency for each operation

**Tips:**
- Sort by duration to find slow research sessions
- Filter by status to locate errors
- Use the timeline view to identify sequential bottlenecks

## 5. Query Traces Programmatically

In [ ]:
import mlflow

traces = mlflow.search_traces(experiment_ids=[exp.experiment_id if exp else "0"])
if len(traces) > 0:
    print(f"Found {len(traces)} traces\n")
    for _, trace in traces.head(5).iterrows():
        print(f"  Trace: {trace.get('request_id', 'N/A')}")
        print(f"  Status: {trace.get('status', 'N/A')}")
        print(f"  Duration: {trace.get('execution_duration', 'N/A')}ms")
        print()
else:
    print("No traces found yet \u2014 run a research query first")
print("\u2705 Trace query complete")

## 6. Span Structure

A typical research session produces the following span tree:

```
[LangGraph] research_session
  \u251c\u2500\u2500 [normalize]
  \u251c\u2500\u2500 [plan] \u2192 ChatOpenAI (LLM call)
  \u251c\u2500\u2500 [execute]
  \u2502     \u251c\u2500\u2500 semantic_search (MCP tool)
  \u2502     \u251c\u2500\u2500 web_search (SearXNG tool)
  \u2502     \u2514\u2500\u2500 draft_report \u2192 ChatOpenAI
  \u251c\u2500\u2500 [verify] \u2192 ChatOpenAI (LLM-as-Judge)
  \u251c\u2500\u2500 [observe]
  \u2514\u2500\u2500 [finalize]
```

Each span records:
- **Start/end timestamps** \u2014 for duration calculations
- **Input/output** \u2014 full payloads (prompts, tool args, responses)
- **Attributes** \u2014 model name, token counts, status codes
- **Parent span ID** \u2014 linking the full call hierarchy

## 7. Summary

In [ ]:
print("\u2705 MLflow tracing walkthrough complete")
print()
print("What we covered:")
print("  1. Connected to MLflow tracking server")
print("  2. Understood autolog tracing (zero code changes)")
print("  3. Explored traces via the MLflow UI")
print("  4. Queried traces programmatically")
print("  5. Reviewed the span tree structure for research sessions")
print()
print("Next: 5_llm_as_judge.ipynb \u2014 Agent evaluation with MLflow Prompt Registry and LLM-as-Judge")